# Aritmética de Complejidad y Análisis de Bucles

El propósito de esta sesión es formalizar las reglas matemáticas que rigen la suma y multiplicación de clases de complejidad, y aplicar modelado sumatorio para determinar el límite asintótico estricto de bucles anidados con dependencias de variables.

Para establecer el problema base de esta sesión, analice el siguiente código en C++ moderno. Este fragmento ilustra la diferencia entre la ejecución secuencial y la ejecución anidada, lo cual servirá como nuestro objeto de estudio matemático.

In [1]:
#include <iostream>
#include <vector>

// Utilizamos std::size_t para garantizar que el índice pueda representar 
// el tamaño máximo de la estructura de datos en la memoria del sistema.
void analizarEstructurasBase(const std::vector<int>& datos) {
    const std::size_t n = datos.size();
    
    if (n == 0) return; // Validación de frontera (O(1))

    // ---------------------------------------------------------
    // Bloque A: Bucle Secuencial
    // ---------------------------------------------------------
    long long suma_total = 0;
    for (std::size_t i = 0; i < n; ++i) {
        // Acceso secuencial a memoria contigua. 
        // Costo constante c_1 dominado por cache prefetching.
        suma_total += datos[i]; 
    }

    // ---------------------------------------------------------
    // Bloque B: Bucles Anidados con Dependencia de Índice
    // ---------------------------------------------------------
    long long pares_procesados = 0;
    for (std::size_t i = 0; i < n; ++i) {
        // Note que el iterador 'j' se inicializa en 'i', creando una 
        // progresión aritmética en el número de ejecuciones internas.
        for (std::size_t j = i; j < n; ++j) {
            // Operación de costo constante c_2
            pares_procesados++; 
        }
    }

    std::cout << "Suma: " << suma_total 
              << ", Pares: " << pares_procesados << "\n";
}

* ***Gestión de Memoria y Tipos***: Utilizamos `std::size_t` en lugar de `int` convencional. En C++, `std::size_t` es un tipo entero sin signo que refleja la capacidad de direccionamiento de memoria del sistema (usualmente 64 bits en arquitecturas modernas). Esto previene overflows al iterar sobre contenedores de la STL (Standard Template Library) como `std::vector`.

* ***Planteamiento del Problema Analítico***: El tiempo de ejecución total $T(n)$ de la función es la suma del tiempo del Bloque A, $T_A(n)$, y el Bloque B, $T_B(n)$. Instintivamente sabemos que $T_A(n) \in \mathcal{O}(n)$. Para $T_B(n)$, el bucle interno no se ejecuta $n$ veces en cada iteración, sino $n-i$ veces.La incógnita que resolveremos es: ¿Cómo justificamos matemáticamente que $T(n) = T_A(n) + T_B(n)$ colapsa en la clase de complejidad del término dominante? ¿Y cuál es el valor exacto polinomial de $T_B(n)$ evaluando la sumatoria $\sum_{i=0}^{n-1} \sum_{j=i}^{n-1} c_2$?

In [2]:
void evaluarInteracciones(const std::vector<double>& particulas) {
    const std::size_t n = particulas.size();
    if (n < 2) return;

    long long comparaciones_ingenuas = 0;
    long long comparaciones_optimizadas = 0;

    // ---------------------------------------------------------
    // Enfoque 1: Recorrido Completo (Matriz N x N)
    // ---------------------------------------------------------
    for (std::size_t i = 0; i < n; ++i) {
        for (std::size_t j = 0; j < n; ++j) {
            if (i != j) { 
                comparaciones_ingenuas++;
            }
        }
    }

    // ---------------------------------------------------------
    // Enfoque 2: Recorrido Triangular Superior (Excluyendo diagonal)
    // ---------------------------------------------------------
    for (std::size_t i = 0; i < n; ++i) {
        for (std::size_t j = i + 1; j < n; ++j) {
            comparaciones_optimizadas++;
        }
    }

    std::cout << "Operaciones (Ingenuo): " << comparaciones_ingenuas << "\n"
              << "Operaciones (Óptimo):  " << comparaciones_optimizadas << "\n";
}

Es común que los desarrolladores asuman que cualquier par de bucles anidados resulta en una complejidad de $\Theta(n^2)$ y, por ende, traten todos los algoritmos cuadráticos como equivalentes. Esta simplificación excesiva ignora la aritmética subyacente y los factores constantes que dictan el rendimiento real en la jerarquía de memoria del hardware.

Para motivar la necesidad de un análisis sumatorio riguroso, examinemos un problema clásico: el cálculo de distancias o interacciones entre todos los pares únicos de elementos en un conjunto (por ejemplo, detección de colisiones en simulaciones físicas o cálculo de varianzas).


In [3]:
#include <chrono>

// Utilidad para medir el tiempo de ejecución en microsegundos
struct Timer {
    std::chrono::high_resolution_clock::time_point start;
    Timer() : start(std::chrono::high_resolution_clock::now()) {}
    
    double elapsed_ms() const {
        auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::milli>(end - start).count();
    }
};

// Función para ejecutar el benchmark comparativo
void ejecutarBenchmarkEmpirico(std::size_t n) {
    // Inicializamos un vector con 'n' elementos.
    std::vector<double> particulas(n, 1.0); 

    std::cout << "--- Iniciando Benchmark con N = " << n << " ---\n\n";

    // ---------------------------------------------------------
    // Instrumentación del Enfoque 1 (O(N^2) con constante 1)
    // ---------------------------------------------------------
    long long iteraciones_1 = 0;
    double suma_dummy_1 = 0.0; // Evita que el compilador elimine el bucle (Dead Code Elimination)

    Timer time_1;
    for (std::size_t i = 0; i < n; ++i) {
        for (std::size_t j = 0; j < n; ++j) {
            if (i != j) {
                iteraciones_1++;
                suma_dummy_1 += particulas[i] * particulas[j];
            }
        }
    }
    auto tiempo_1 = time_1.elapsed_ms();
    

    // ---------------------------------------------------------
    // Instrumentación del Enfoque 2 (O(N^2) con constante 1/2)
    // ---------------------------------------------------------
    long long iteraciones_2 = 0;
    double suma_dummy_2 = 0.0;    // Evita que el compilador elimine el bucle (Dead Code Elimination)

    Timer time_2;
    for (std::size_t i = 0; i < n; ++i) {
        for (std::size_t j = i + 1; j < n; ++j) {
            iteraciones_2++;
            suma_dummy_2 += particulas[i] * particulas[j];
        }
    }
    auto tiempo_2 = time_2.elapsed_ms();
    

    // ---------------------------------------------------------
    // Análisis de Resultados
    // ---------------------------------------------------------
    std::cout << "[Enfoque 1 - Completo]\n"
              << "Iteraciones: " << iteraciones_1 << "\n"
              << "Tiempo real: " << tiempo_1 << " ms\n\n";

    std::cout << "[Enfoque 2 - Triangular]\n"
              << "Iteraciones: " << iteraciones_2 << "\n"
              << "Tiempo real: " << tiempo_2 << " ms\n\n";

    double ratio_tiempo = tiempo_2 / tiempo_1;
    std::cout << "Ratio Empírico de Tiempo (T2 / T1): " << ratio_tiempo << "\n";
    
    // Imprimimos los resultados para garantizar la ejecución de los cálculos
    // (Previene optimizaciones agresivas del nivel -O3)
    std::cout << "(Validación de estado: " << suma_dummy_1 << ", " << suma_dummy_2 << ")\n";
}

void test() {
    const std::size_t N = 10'000;
    // 10,000 elementos representan 100 millones de posibles interacciones
    ejecutarBenchmarkEmpirico(N);
}

test();

--- Iniciando Benchmark con N = 10000 ---

[Enfoque 1 - Completo]
Iteraciones: 99990000
Tiempo real: 480.252 ms

[Enfoque 2 - Triangular]
Iteraciones: 49995000
Tiempo real: 217.806 ms

Ratio Empírico de Tiempo (T2 / T1): 0.453524
(Validación de estado: 9.999e+07, 4.9995e+07)


Desde una perspectiva superficial, ambos enfoques pertenecen a la clase de complejidad $\mathcal{O}(n^2)$ aplicando la regla del término dominante (como discutimos, evaluando el máximo de las funciones envolventes). Sin embargo, en la práctica:

* El Enfoque 1 ejecuta exactamente $n^2$ iteraciones del bucle interno, introduciendo una instrucción de salto condicional (`if (i != j)`) que puede causar branch mispredictions en el pipeline del procesador, degradando severamente los ciclos de reloj por instrucción (CPI).

* El Enfoque 2 reestructura el espacio de iteración. Matemáticamente, el número de iteraciones no es $n^2$, sino que obedece a la serie aritmética: $\sum_{i=0}^{n-1} (n - 1 - i)$.

La diferencia empírica es que el Enfoque 2 ejecuta exactamente $\frac{n(n-1)}{2}$ operaciones. Aunque $\frac{1}{2}n^2 - \frac{1}{2}n \in \Theta(n^2)$, el factor constante de $\frac{1}{2}$ y la eliminación del costo condicional (operación pura en la ALU) significan que el Enfoque 2 será consistentemente más del doble de rápido en tiempo de pared (wall-clock time).Entender cómo aislar matemáticamente ese factor $\frac{1}{2}$ y demostrar que el algoritmo es estrictamente $\Theta(n^2)$ requiere aritmética de sumatorias, no solo reglas empíricas.

# Perfilador Universal

En lugar de repetir el código de medición para cada enfoque, utilizamos las capacidades de metaprogramación de plantillas de C++ moderno.

In [4]:
#include <iostream>
#include <vector>
#include <chrono>
#include <utility>
#include <iomanip>

// ---------------------------------------------------------
// Perfilador Universal usando Plantillas Variádicas (C++17)
// ---------------------------------------------------------
template <typename Func, typename... Args>
double perfilarAlgoritmo(Func&& func, Args&&... args) {
    auto inicio = std::chrono::high_resolution_clock::now();
    
    // std::forward propaga las referencias (lvalue) correctamente, 
    // permitiéndonos modificar la variable de 'resultado' externa.
    std::forward<Func>(func)(std::forward<Args>(args)...);
    
    auto fin = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> tiempo = fin - inicio;
    
    return tiempo.count();
}

// =========================================================
// LAS 3 IMPLEMENTACIONES DEL MISMO ALGORITMO
// =========================================================

// [1] Enfoque Ingenuo - O(N^2)
// Itera sobre la matriz completa N x N. Penalizado severamente por
// saltos condicionales y operaciones redundantes.
void sumarParesIngenuo(const std::vector<int>& datos, long long& resultado) {
    long long suma = 0;
    std::size_t n = datos.size();
    for (std::size_t i = 0; i < n; ++i) {
        for (std::size_t j = 0; j < n; ++j) {
            if (i != j) {
                // 1LL fuerza la promoción a 64 bits para evitar overflow
                suma += 1LL * datos[i] * datos[j]; 
            }
        }
    }
    // Divide por 2 porque calculó cada par dos veces (ej. A*B y B*A)
    resultado = suma / 2; 
}

// [2] Enfoque Triangular Superior - O(N^2)
// Reduce a la mitad las operaciones teóricas y elimina el "branching"
// (Simpatía Mecánica). El hardware operará al máximo de su capacidad.
void sumarParesTriangular(const std::vector<int>& datos, long long& resultado) {
    long long suma = 0;
    std::size_t n = datos.size();
    for (std::size_t i = 0; i < n; ++i) {
        for (std::size_t j = i + 1; j < n; ++j) {
            suma += 1LL * datos[i] * datos[j];
        }
    }
    resultado = suma;
}

// [3] Enfoque Algebraico - O(N)
// Utiliza la identidad matemática: Suma(a_i * a_j) = [(Suma a_i)^2 - Suma(a_i^2)] / 2
// Destruye la barrera de la complejidad cuadrática reduciendo la matriz a vectores.
void sumarParesLineal(const std::vector<int>& datos, long long& resultado) {
    long long suma_lineal = 0;
    long long suma_cuadrados = 0;
    for (int val : datos) {
        suma_lineal += val;
        suma_cuadrados += 1LL * val * val;
    }
    resultado = (suma_lineal * suma_lineal - suma_cuadrados) / 2;
}

// =========================================================
// BLOQUE DE EJECUCIÓN
// =========================================================
void prueba() {
    // Reducimos N a 50,000 para que el enfoque Ingenuo no tome tanto tiempo
    // 50,000 elementos implican evaluar ~500 millones de interacciones.
    const std::size_t N = 50'000;
    std::vector<int> datos_prueba(N, 2); 
    
    std::cout << "--- Benchmark de 3 Implementaciones (N = " << N << ") ---\n\n";

    long long res1 = 0, res2 = 0, res3 = 0;

    // Ejecutamos e imprimimos resultados
    double ms1 = perfilarAlgoritmo(sumarParesIngenuo, datos_prueba, res1);
    std::cout << "[1] Ingenuo    O(N^2) | Tiempo: " 
              << std::fixed << std::setprecision(3) << ms1 << " ms \t| Resultado: " << res1 << "\n";

    double ms2 = perfilarAlgoritmo(sumarParesTriangular, datos_prueba, res2);
    std::cout << "[2] Triangular O(N^2) | Tiempo: " 
              << ms2 << " ms \t| Resultado: " << res2 << "\n";

    double ms3 = perfilarAlgoritmo(sumarParesLineal, datos_prueba, res3);
    std::cout << "[3] Algebraico O(N)   | Tiempo: " 
              << ms3 << " ms \t| Resultado: " << res3 << "\n";
}


prueba();


--- Benchmark de 3 Implementaciones (N = 50000) ---

[1] Ingenuo    O(N^2) | Tiempo: 12827.832 ms 	| Resultado: 4999900000
[2] Triangular O(N^2) | Tiempo: 5919.133 ms 	| Resultado: 4999900000
[3] Algebraico O(N)   | Tiempo: 0.360 ms 	| Resultado: 4999900000
